In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

In [2]:
def split_and_save_dataset(
    df, 
    output_dir='dataset_splits', 
    test_size=0.15, 
    val_size=0.15, 
    random_state=42,
    prefix='data'
):
    """
    Split dataset and save to CSV files
    
    Parameters:
    - df: Input DataFrame
    - output_dir: Directory to save split datasets
    - test_size: Proportion of dataset to include in test split
    - val_size: Proportion of dataset to include in validation split
    - random_state: Random seed for reproducibility
    - prefix: Prefix for output filenames
    
    Returns:
    - Paths to saved CSV files
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # First, split off the test set while stratifying by ticker
    train_val_df, test_df = train_test_split(
        df, 
        test_size=test_size, 
        stratify=df['ticker'], 
        random_state=random_state
    )
    
    # Now split the remaining data into train and validation sets
    # Adjust validation size relative to the remaining dataset
    val_proportion = val_size / (1 - test_size)
    train_df, val_df = train_test_split(
        train_val_df, 
        test_size=val_proportion, 
        stratify=train_val_df['ticker'], 
        random_state=random_state
    )
    
    # Prepare file paths
    train_path = os.path.join(output_dir, f'{prefix}_train.csv')
    val_path = os.path.join(output_dir, f'{prefix}_val.csv')
    test_path = os.path.join(output_dir, f'{prefix}_test.csv')
    
    # Save datasets to CSV
    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path, index=False)
    test_df.to_csv(test_path, index=False)
    
    # Verification print
    print("Datasets split and saved:")
    print(f"Train: {train_path} - {len(train_df)} samples")
    print(f"Validation: {val_path} - {len(val_df)} samples")
    print(f"Test: {test_path} - {len(test_df)} samples")
    
    # Verify ticker distribution
    print("\nTicker Distribution:")
    print("Train Tickers:\n", train_df['ticker'].value_counts(normalize=True))
    print("\nValidation Tickers:\n", val_df['ticker'].value_counts(normalize=True))
    print("\nTest Tickers:\n", test_df['ticker'].value_counts(normalize=True))
    
    return train_path, val_path, test_path


In [3]:
df = pd.read_parquet("hf://datasets/virattt/financial-qa-10K/data/train-00000-of-00001.parquet")
df.head()

,question,answer,context,ticker,filing
0,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,"Since our original focus on PC graphics, we ha...",NVDA,2023_10K
1,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,Some of the most recent applications of GPU-po...,NVDA,2023_10K
2,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,Our invention of the GPU in 1999 defined moder...,NVDA,2023_10K
3,How does NVIDIA's platform strategy contribute...,NVIDIA's platform strategy brings together har...,"NVIDIA has a platform strategy, bringing toget...",NVDA,2023_10K
4,What does NVIDIA's CUDA programming model enable?,NVIDIA's CUDA programming model opened the par...,With our introduction of the CUDA programming ...,NVDA,2023_10K


In [4]:
split_and_save_dataset(df)

Datasets split and saved:
Train: dataset_splits\data_train.csv - 4900 samples
Validation: dataset_splits\data_val.csv - 1050 samples
Test: dataset_splits\data_test.csv - 1050 samples

Ticker Distribution:
Train Tickers:
 ticker
JNJ      0.028571
AXP      0.014286
HPE      0.014286
ETSY     0.014286
CMCSA    0.014286
           ...   
GME      0.014286
WMT      0.014286
AMC      0.014286
CVX      0.014286
CAT      0.014286
Name: proportion, Length: 69, dtype: float64

Validation Tickers:
 ticker
JNJ      0.028571
JPM      0.014286
BRK-A    0.014286
PTON     0.014286
NVDA     0.014286
           ...   
GILD     0.014286
GME      0.014286
GS       0.014286
HPE      0.014286
LULU     0.014286
Name: proportion, Length: 69, dtype: float64

Test Tickers:
 ticker
JNJ     0.028571
MSFT    0.014286
CB      0.014286
T       0.014286
BAC     0.014286
          ...   
AMD     0.014286
PLTR    0.014286
GIS     0.014286
GRMN    0.014286
HD      0.014286
Name: proportion, Length: 69, dtype: float64


('dataset_splits\\data_train.csv',
 'dataset_splits\\data_val.csv',
 'dataset_splits\\data_test.csv')

In [6]:
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

In [10]:
financial_prompt = """
You are a financial advisor analyzing stock prices based on news and market data. Your task is to answer user questions by interpreting stock movements and providing actionable advice.

For each query, provide:
1. A prediction of the stock's direction (up, down, or neutral) based on available news or trends.
2. Explain the reasoning behind your prediction, referencing news, data, or trends.
3. Suggest an action (buy, sell, hold, or watch closely), or state if unsure.
4. If uncertain, say "I don't have enough data to predict."
5. If speculative, say “This is a hypothetical scenario.”

Example 1:
User: "What do you think about Tesla stock next week?"
Answer: "Tesla's earnings exceeded expectations, so the stock might rise. I recommend holding or buying if entering, but watch for new product announcements."

Example 2:
User: "Should I buy AAPL after the new iPhone release?"
Answer: "The iPhone release could boost AAPL short-term. Hold if long-term, but short-term traders should wait for sales data."

Example 3:
User: "What should I do with Microsoft after the market crash?"
Answer: "Microsoft tends to recover post-crash. Hold for the long term or watch for signs of recovery in the short term."

Example 4:
User: "Should I invest in a new AI startup?"
Answer: "Investing in a new AI startup is high risk. Consider a small investment if you're willing to take on risk, but hold off if you're conservative."

Example 5 (when unsure):
User: "What will happen to Amazon stock in the next month?"
Answer: "I don't have enough data to predict. Keep an eye on Amazon's upcoming earnings report and market news for more clarity."
"""


In [33]:
import json
import numpy as np

def handle_nan(value):
    """
    Convert NaN values to None or other appropriate representations
    """
    if pd.isna(value):
        return None
    elif isinstance(value, (np.integer, np.floating)):
        return value.item()
    return value

def csv_to_jsonl(input_csv_path, output_dir=None, dropna=False, fillna=None):
    """
    Convert a CSV file to JSONL format with NaN handling
    
    Parameters:
    - input_csv_path: Path to the input CSV file
    - output_dir: Directory to save the output JSONL file
    - dropna: Whether to drop rows with NaN values
    - fillna: Value to fill NaN with (optional)
    
    Returns:
    - Path to the saved JSONL file
    """
    # Read the CSV file
    df = pd.read_csv(input_csv_path)
    
    # Check and handle NaN values
    print("\nNaN Value Analysis:")
    print(df.isna().sum())
    
    # Option to drop NaN rows
    if dropna:
        df_processed = df.dropna()
        print(f"\nDropped {len(df) - len(df_processed)} rows with NaN values")
    # Option to fill NaN
    elif fillna is not None:
        df_processed = df.fillna(fillna)
        print(f"\nFilled NaN values with: {fillna}")
    else:
        df_processed = df
    
    # Determine output directory
    if output_dir is None:
        output_dir = os.path.dirname(input_csv_path)
    os.makedirs(output_dir, exist_ok=True)
    
    # Create output filename
    base_filename = os.path.splitext(os.path.basename(input_csv_path))[0]
    output_jsonl_path = os.path.join(output_dir, f"{base_filename}.jsonl")
    
    # Convert each row to a dictionary
    with open(output_jsonl_path, 'w') as jsonl_file:
        for _, row in df_processed.iterrows():
            # Convert row to dictionary with NaN handling
            row_dict = {
                col: handle_nan(value) 
                for col, value in row.items()
            }

            base_messages = {"messages": [
            {"role": "user", "content": row_dict["question"]}, 
            {"role": "assistant", "content": row_dict["answer"]}]}
            
            # Write each row as a JSON line
            json.dump(base_messages, jsonl_file)
            jsonl_file.write('\n')
    
    print(f"\nJSONL file saved to: {output_jsonl_path}")
    print(f"Total entries: {len(df_processed)}")
    
    return output_jsonl_path

def convert_csvs_to_jsonl(input_dir, output_dir=None, **kwargs):
    """
    Convert all CSV files in a directory to JSONL with NaN handling
    
    Parameters:
    - input_dir: Directory containing CSV files
    - output_dir: Directory to save JSONL files
    - **kwargs: Additional arguments for csv_to_jsonl (dropna, fillna)
    
    Returns:
    - List of paths to created JSONL files
    """
    # If no output directory specified, create a 'jsonl' subdirectory
    if output_dir is None:
        output_dir = os.path.join(input_dir, 'jsonl')
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Find all CSV files
    csv_files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]
    
    # Store paths of created JSONL files
    jsonl_files = []
    
    for csv_file in csv_files:
        # Construct full input path
        input_path = os.path.join(input_dir, csv_file)
        
        # Convert the CSV to JSONL with NaN handling
        jsonl_path = csv_to_jsonl(input_path, output_dir, **kwargs)
        jsonl_files.append(jsonl_path)
    
    print(f"\nConverted {len(csv_files)} CSV files to JSONL in {output_dir}")
    return jsonl_files

In [34]:
convert_csvs_to_jsonl('dataset_splits', fillna='Missing data')


NaN Value Analysis:
question    0
answer      1
context     0
ticker      0
filing      0
dtype: int64

Filled NaN values with: Missing data

JSONL file saved to: dataset_splits\jsonl\data_test.jsonl
Total entries: 1050

NaN Value Analysis:
question    1
answer      0
context     0
ticker      0
filing      0
dtype: int64

Filled NaN values with: Missing data

JSONL file saved to: dataset_splits\jsonl\data_train.jsonl
Total entries: 4900

NaN Value Analysis:
question    1
answer      1
context     1
ticker      0
filing      0
dtype: int64

Filled NaN values with: Missing data

JSONL file saved to: dataset_splits\jsonl\data_val.jsonl
Total entries: 1050

Converted 3 CSV files to JSONL in dataset_splits\jsonl


['dataset_splits\\jsonl\\data_test.jsonl',
 'dataset_splits\\jsonl\\data_train.jsonl',
 'dataset_splits\\jsonl\\data_val.jsonl']

In [35]:
from openai import OpenAI
client = OpenAI()

training_file = client.files.create(
  file=open("dataset_splits/jsonl/data_train.jsonl", "rb"),
  purpose="fine-tune"
)

validation_file = client.files.create(
  file=open("dataset_splits/jsonl/data_val.jsonl", "rb"),
  purpose="fine-tune"
)

In [36]:
print("Training file ID:", training_file.id)
print("Validation file ID:", validation_file.id)

Training file ID: file-99r1A2ySyE9rTpNp2WvPC6
Validation file ID: file-3v3ZwE4GHCCzSW6VcBxj6G


In [37]:
response = client.fine_tuning.jobs.create(
  training_file=training_file.id,
  validation_file=validation_file.id,
  model="gpt-4o-mini-2024-07-18",
)

In [39]:
print("Fine-tuning job started:", response)

Fine-tuning job started: FineTuningJob(id='ftjob-xfEdZpRnB6Jh8T1OKaLIBNf0', created_at=1734203845, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs='auto', batch_size='auto', learning_rate_multiplier='auto'), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-jLegqlpVUK2GSLiL6tESx6hy', result_files=[], seed=12637339, status='validating_files', trained_tokens=None, training_file='file-99r1A2ySyE9rTpNp2WvPC6', validation_file='file-3v3ZwE4GHCCzSW6VcBxj6G', estimated_finish=None, integrations=[], user_provided_suffix=None)


In [7]:
!openai api models.list

{
  "id": "text-embedding-3-small",
  "created": 1705948997,
  "object": "model",
  "owned_by": "system"
}
{
  "id": "gpt-4o",
  "created": 1715367049,
  "object": "model",
  "owned_by": "system"
}
{
  "id": "gpt-4o-realtime-preview",
  "created": 1727659998,
  "object": "model",
  "owned_by": "system"
}
{
  "id": "dall-e-2",
  "created": 1698798177,
  "object": "model",
  "owned_by": "system"
}
{
  "id": "gpt-4o-realtime-preview-2024-10-01",
  "created": 1727131766,
  "object": "model",
  "owned_by": "system"
}
{
  "id": "o1-mini-2024-09-12",
  "created": 1725648979,
  "object": "model",
  "owned_by": "system"
}
{
  "id": "gpt-4-1106-preview",
  "created": 1698957206,
  "object": "model",
  "owned_by": "system"
}
{
  "id": "o1-mini",
  "created": 1725649008,
  "object": "model",
  "owned_by": "system"
}
{
  "id": "gpt-3.5-turbo-instruct",
  "created": 1692901427,
  "object": "model",
  "owned_by": "system"
}
{
  "id": "babbage-002",
  "created": 1692634615,
  "object": "model",
  "own